# Aula 1 - Testes Automatizados para Modelos de IA
## Testes unitários, integração e regressão para modelos.
## Estratégias de teste para aplicações de ML.

## Setup

Usamos o [`ipytest`](https://github.com/chmp/ipytest) para rodar testes `pytest` de dentro do notebook, com o mesmo relatório que você veria no terminal.

In [ ]:
!pip install -q "ipytest==0.14.*"

import ipytest
import pytest
ipytest.autoconfig()

---
## 1. Testes unitários com pytest

Vamos testar uma função simples de pré-processamento de texto: normalizar espaços e caixa.

In [ ]:
%%ipytest

def preprocess(text: str) -> str:
    return text.strip().lower()


def test_preprocess_removes_whitespace_and_lowercases():
    assert preprocess("  Hello WORLD  ") == "hello world"


### Exercício

Escreva **2 casos de teste** para `preprocess`, decididos por vocês. Pelo menos **1 deles precisa ser um caso-limite** — cabe a vocês identificar o que conta como caso-limite pra essa função, não vamos listar exemplos aqui. Complete a célula abaixo: escolha o caso, renomeie a função para descrever o que ela verifica, e escreva o assert.

In [ ]:
%%ipytest

def preprocess(text: str) -> str:
    return text.strip().lower()


# Casos escolhidos (não cobertos pelo exemplo "  Hello WORLD  "):
# 1. texto já sem espaços extras — só precisa de lower()
# 2. caso-limite: string vazia (entrada "nada")

def test_preprocess_lowercases_without_extra_spaces():
    assert preprocess("Machine Learning") == "machine learning"


def test_preprocess_empty_string():
    assert preprocess("") == ""


---
## 2. Teste de integração de pipeline

Um teste de integração verifica se as etapas do pipeline funcionam corretamente **encadeadas**. Aqui, encadeamos `preprocess` com um modelo mock.

In [ ]:
%%ipytest

import re


def preprocess(text: str) -> str:
    return text.strip().lower()


class MockModel:
    """Modelo mock: simula um classificador de sentimento bem simples,
    só para fins didáticos (sem treinar nada de verdade)."""

    POSITIVE_WORDS = {"great", "good", "excellent", "hello"}
    NEGATIVE_WORDS = {"terrible", "bad", "awful"}

    def predict(self, text: str) -> str:
        words = set(re.findall(r"[a-z]+", text))
        if words & self.POSITIVE_WORDS:
            return "positive"
        if words & self.NEGATIVE_WORDS:
            return "negative"
        return "neutral"


mock_model = MockModel()


def test_pipeline_end_to_end():
    raw_input = "  Hello WORLD  "
    processed = preprocess(raw_input)
    prediction = mock_model.predict(processed)
    assert prediction in ["positive", "negative", "neutral"]


---
## 3. Teste de regressão com golden dataset

Um golden dataset é um conjunto de casos com resultado esperado conhecido. Pode ser aplicado para detectar se uma nova versão do modelo passou a errar algo que já funcionava.

Aqui cada caso do golden dataset vira um teste independente via @pytest.mark.parametrize - assim o relatório mostra individualmente quais casos passaram e quais falharam, em vez de um único teste que para na primeira falha.

In [ ]:
%%ipytest

import re


def preprocess(text: str) -> str:
    return text.strip().lower()


class MockModel:
    POSITIVE_WORDS = {"great", "good", "excellent", "hello"}
    NEGATIVE_WORDS = {"terrible", "bad", "awful"}

    def predict(self, text: str) -> str:
        words = set(re.findall(r"[a-z]+", text))
        if words & self.POSITIVE_WORDS:
            return "positive"
        if words & self.NEGATIVE_WORDS:
            return "negative"
        return "neutral"


mock_model = MockModel()

GOLDEN_CASES = [
    ("great product!", "positive"),
    ("terrible experience", "negative"),
]


@pytest.mark.parametrize("text, expected", GOLDEN_CASES)
def test_no_regression_on_golden_cases(text, expected):
    assert mock_model.predict(preprocess(text)) == expected


### Exercício

Pensando num domínio hipotético (ex: e-commerce, suporte técnico), adicione **2 a 3 casos** novos a `GOLDEN_CASES_EXTRA` abaixo.

**Desafio:** pelo menos 1 dos seus casos deve ser um texto onde você *desconfia* que o `MockModel` vai errar ou dar uma resposta duvidosa. Não vale só repetir uma palavra que já está em `POSITIVE_WORDS`/`NEGATIVE_WORDS` (definidas acima) — o objetivo é achar o limite da lógica do modelo, não confirmar o óbvio. Anote o resultado que você espera e compare com o que o modelo realmente devolve quando rodar a célula.

In [ ]:
%%ipytest

import re


def preprocess(text: str) -> str:
    return text.strip().lower()


class MockModel:
    POSITIVE_WORDS = {"great", "good", "excellent", "hello"}
    NEGATIVE_WORDS = {"terrible", "bad", "awful"}

    def predict(self, text: str) -> str:
        words = set(re.findall(r"[a-z]+", text))
        if words & self.POSITIVE_WORDS:
            return "positive"
        if words & self.NEGATIVE_WORDS:
            return "negative"
        return "neutral"


mock_model = MockModel()

# Domínio: e-commerce / suporte.
# expected = rótulo que um humano esperaria (não o "óbvio" do dicionário).
# Ao rodar: compare expected x saída real do MockModel.
GOLDEN_CASES_EXTRA = [
    # Negação: humano espera negative; modelo vê "good" e devolve positive.
    ("Isso não é bom de jeito nenhum", "negative"),
    # Vocabulário fora da lista (quebrado/reembolso): humano espera negative; modelo devolve neutral.
    ("Produto chegou quebrado, quero reembolso", "negative"),
    # "hello" na lista positiva: ticket de suporte parece neutral; modelo devolve positive.
    ("Ola, preciso de ajuda com o meu pedido", "neutral"),
]


def test_golden_cases_extra_not_empty():
    assert GOLDEN_CASES_EXTRA, "adicione pelo menos 2 casos antes de rodar"


@pytest.mark.parametrize("text, expected", GOLDEN_CASES_EXTRA)
def test_golden_cases_extra(text, expected):
    assert mock_model.predict(preprocess(text)) == expected


---
## 4. Suíte completa

##Agora juntamos tudo em uma suíte:
##    testes unitários da função de pré-processamento
##    teste de integração do pipeline
##    teste de regressão com golden dataset.
##    
##Rode a célula abaixo e observe o relatório do `pytest`. Avalie quantos testes passaram, quanto tempo levou, e o que aconteceria se um deles falhasse.

In [ ]:
%%ipytest -v

# --- código sob teste -------------------------------------------------

import re


def preprocess(text: str) -> str:
    return text.strip().lower()


class MockModel:
    POSITIVE_WORDS = {"great", "good", "excellent", "hello"}
    NEGATIVE_WORDS = {"terrible", "bad", "awful"}

    def predict(self, text: str) -> str:
        words = set(re.findall(r"[a-z]+", text))
        if words & self.POSITIVE_WORDS:
            return "positive"
        if words & self.NEGATIVE_WORDS:
            return "negative"
        return "neutral"


mock_model = MockModel()

GOLDEN_CASES = [
    ("great product!", "positive"),
    ("terrible experience", "negative"),
]

# --- suíte de testes ----------------------------------------------------

def test_preprocess_removes_whitespace_and_lowercases():
    assert preprocess("  Hello WORLD  ") == "hello world"


def test_preprocess_empty_string():
    assert preprocess("") == ""


def test_pipeline_end_to_end():
    processed = preprocess("  Hello WORLD  ")
    prediction = mock_model.predict(processed)
    assert prediction in ["positive", "negative", "neutral"]


@pytest.mark.parametrize("text, expected", GOLDEN_CASES)
def test_no_regression_on_golden_cases(text, expected):
    assert mock_model.predict(preprocess(text)) == expected


### Demonstração: regressão de modelo

Vamos simular uma nova versão do modelo que esqueceu uma palavra negativa durante um retreino. Rode a célula abaixo e observe o relatório de FALHA do pytest, exatamente esse tipo de sinal vermelho que o teste de regressão existe para dar.

Como o golden dataset está parametrizado, o relatório mostra cada caso individualmente: reparem que o caso great product! continua PASSED e só terrible experience aparece como FAILED - e exatamente esse caso, isolado, que aponta a regressão.

In [ ]:
%%ipytest -v

import re


def preprocess(text: str) -> str:
    return text.strip().lower()


class MockModelV2:
    """Versão quebrada: o retreino removeu terrible do vocabulário negativo."""

    POSITIVE_WORDS = {"great", "good", "excellent", "hello"}
    NEGATIVE_WORDS = {"bad", "awful"}  # terrible sumiu daqui

    def predict(self, text: str) -> str:
        words = set(re.findall(r"[a-z]+", text))
        if words & self.POSITIVE_WORDS:
            return "positive"
        if words & self.NEGATIVE_WORDS:
            return "negative"
        return "neutral"


mock_model_v2 = MockModelV2()

GOLDEN_CASES = [
    ("great product!", "positive"),
    ("terrible experience", "negative"),
]


@pytest.mark.parametrize("text, expected", GOLDEN_CASES)
def test_no_regression_on_golden_cases_v2(text, expected):
    assert mock_model_v2.predict(preprocess(text)) == expected


### Discussão

Num sistema de recomendação de produtos, qual desses três tipos de teste (unitário, integração, regressão) vocês priorizariam primeiro, e por quê?